# ZGLMM384 — Monitor de requisicao de compra (RC -> pedido) — Análise Exploratória

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra`
**Transação SAP:** ZGLMM384
**Colunas:** 67 · **Clustering declarado:** `cod_requisicao_compra`, `cod_material`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória entre elas.

## Alertas de partida _(extraídos dos comentários do `DESCRIBE`)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | **Clustering não inclui o item.** Está em `cod_requisicao_compra + cod_material`, mas os comentários marcam a PK como `cod_requisicao_compra + cod_item_requisicao_compra`. Uma RC com vários itens quebra a unicidade do clustering — ver seção 4. |
| 🟡 MEDIA | **5 colunas `boolean`.** O SAP exporta indicadores como `X`/vazio; aqui virou `true`/`false`. É conversão de tipo, não divergência de valor. |
| 🟡 MEDIA | **`vl_requisicao_compra` é `double`** enquanto as quantidades usam `decimal(13,3)` — aplicar tolerância de 0,005 na conciliação. |
| ⚪ BAIXA | **26 colunas sem comentário** no `DESCRIBE` — candidatas a nunca carregadas (seção 6). |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Flags booleanas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas |
| 12.3 | Timestamps |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| **18** | **Análises específicas da transação** |
| 19 | Amostra ampliada |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra;

In [0]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra;

In [0]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra LIMIT 20;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                                        AS linhas,
       COUNT(DISTINCT CONCAT_WS('|', `cod_requisicao_compra`, `cod_item_requisicao_compra`, `num_pedido_compra`, `num_item_pedido_compra`)) AS chaves_completas,
       COUNT(DISTINCT `cod_centro`)                                 AS valores_de_cod_centro,
       COUNT(DISTINCT `cod_requisicao_compra`)                                 AS cod_requisicao_compra_distintos
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas)
que contenha os casos-limite identificados nas demais seções.

In [0]:
-- 3. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra),
g AS (
  SELECT 'cod_requisicao_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_requisicao_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
UNION ALL
  SELECT 'cod_requisicao_compra + cod_item_requisicao_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_requisicao_compra`, `cod_item_requisicao_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
UNION ALL
  SELECT 'cod_requisicao_compra + cod_item_requisicao_compra + num_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_requisicao_compra`, `cod_item_requisicao_compra`, `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
UNION ALL
  SELECT 'cod_requisicao_compra + cod_item_requisicao_compra + num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_requisicao_compra`, `cod_item_requisicao_compra`, `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `cod_requisicao_compra + cod_material`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_requisicao_compra`, `cod_material`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY `cod_requisicao_compra`, `cod_material`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (
  SELECT `cod_requisicao_compra`, `cod_material` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `cod_requisicao_compra`, `cod_material` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra c JOIN dup ON c.`cod_requisicao_compra` <=> dup.`cod_requisicao_compra` AND c.`cod_material` <=> dup.`cod_material`
),
agg AS (
  SELECT `cod_requisicao_compra`, `cod_material`,
         COUNT(DISTINCT `cod_item_requisicao_compra`) AS `cod_item_requisicao_compra`,
         COUNT(DISTINCT `tp_documento_requisicao_compra`) AS `tp_documento_requisicao_compra`,
         COUNT(DISTINCT `st_processamento_requisicao_compra`) AS `st_processamento_requisicao_compra`,
         COUNT(DISTINCT `st_liberacao_requisicao_compra`) AS `st_liberacao_requisicao_compra`,
         COUNT(DISTINCT `ind_requisicao_compra_concluida`) AS `ind_requisicao_compra_concluida`,
         COUNT(DISTINCT `tp_ctg_item_documento_compra`) AS `tp_ctg_item_documento_compra`,
         COUNT(DISTINCT `num_pedido_compra`) AS `num_pedido_compra`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `num_item_pedido_compra`) AS `num_item_pedido_compra`,
         COUNT(DISTINCT `nm_centro`) AS `nm_centro`,
         COUNT(DISTINCT `cod_grupo_compradores`) AS `cod_grupo_compradores`,
         COUNT(DISTINCT `nm_grupo_compradores`) AS `nm_grupo_compradores`,
         COUNT(DISTINCT `cod_organizacao_compras`) AS `cod_organizacao_compras`,
         COUNT(DISTINCT `cod_texto_breve`) AS `cod_texto_breve`,
         COUNT(DISTINCT `desc_texto_longo`) AS `desc_texto_longo`,
         COUNT(DISTINCT `qt_solicitada`) AS `qt_solicitada`,
         COUNT(DISTINCT `sg_medida_pedido`) AS `sg_medida_pedido`,
         COUNT(DISTINCT `tp_grupo_mercadorias`) AS `tp_grupo_mercadorias`,
         COUNT(DISTINCT `vl_requisicao_compra`) AS `vl_requisicao_compra`,
         COUNT(DISTINCT `qt_disponibilidade_estoque`) AS `qt_disponibilidade_estoque`,
         COUNT(DISTINCT `tp_classificacao_contabil`) AS `tp_classificacao_contabil`,
         COUNT(DISTINCT `tp_material`) AS `tp_material`,
         COUNT(DISTINCT `vl_prazo_entrega_previsto_dias`) AS `vl_prazo_entrega_previsto_dias`,
         COUNT(DISTINCT `tp_mrp`) AS `tp_mrp`,
         COUNT(DISTINCT `cod_planejador_mrp`) AS `cod_planejador_mrp`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `cod_elemento_pep`) AS `cod_elemento_pep`,
         COUNT(DISTINCT `cod_diagrama_rede`) AS `cod_diagrama_rede`,
         COUNT(DISTINCT `cod_conta_razao`) AS `cod_conta_razao`,
         COUNT(DISTINCT `num_reserva`) AS `num_reserva`,
         COUNT(DISTINCT `dt_solicitacao_rc`) AS `dt_solicitacao_rc`,
         COUNT(DISTINCT `dt_necessidade_rc`) AS `dt_necessidade_rc`,
         COUNT(DISTINCT `dt_liberacao_requisicao_compra`) AS `dt_liberacao_requisicao_compra`,
         COUNT(DISTINCT `qt_dias_rc`) AS `qt_dias_rc`,
         COUNT(DISTINCT `cod_usuario_criacao`) AS `cod_usuario_criacao`,
         COUNT(DISTINCT `nm_usuario_criacao`) AS `nm_usuario_criacao`,
         COUNT(DISTINCT `num_folha_servico`) AS `num_folha_servico`,
         COUNT(DISTINCT `dt_liberacao_folha`) AS `dt_liberacao_folha`,
         COUNT(DISTINCT `cod_servico`) AS `cod_servico`,
         COUNT(DISTINCT `desc_servico`) AS `desc_servico`,
         COUNT(DISTINCT `sg_medida_servico`) AS `sg_medida_servico`,
         COUNT(DISTINCT `dh_resposta`) AS `dh_resposta`,
         COUNT(DISTINCT `nm_motivo_rejeicao`) AS `nm_motivo_rejeicao`,
         COUNT(DISTINCT `desc_comentarios_fornecedor`) AS `desc_comentarios_fornecedor`,
         COUNT(DISTINCT `num_confirmacao_ordem`) AS `num_confirmacao_ordem`,
         COUNT(DISTINCT `qt_pedido`) AS `qt_pedido`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `dt_pedido`) AS `dt_pedido`,
         COUNT(DISTINCT `cod_usuario_3`) AS `cod_usuario_3`,
         COUNT(DISTINCT `num_contrato`) AS `num_contrato`,
         COUNT(DISTINCT `ind_escopo_guiado`) AS `ind_escopo_guiado`,
         COUNT(DISTINCT `num_item_contrato`) AS `num_item_contrato`,
         COUNT(DISTINCT `dt_parada_inicio`) AS `dt_parada_inicio`,
         COUNT(DISTINCT `dt_parada_fim`) AS `dt_parada_fim`,
         COUNT(DISTINCT `num_referencia`) AS `num_referencia`,
         COUNT(DISTINCT `desc_anotacoes_comprador`) AS `desc_anotacoes_comprador`,
         COUNT(DISTINCT `ind_item_eliminado_rc`) AS `ind_item_eliminado_rc`,
         COUNT(DISTINCT `cod_eliminacao_documento_compras`) AS `cod_eliminacao_documento_compras`,
         COUNT(DISTINCT `ind_pedido_memorizado`) AS `ind_pedido_memorizado`,
         COUNT(DISTINCT `tp_urgencia`) AS `tp_urgencia`,
         COUNT(DISTINCT `ind_item_cotacao`) AS `ind_item_cotacao`,
         COUNT(DISTINCT `dt_envio_portal`) AS `dt_envio_portal`,
         COUNT(DISTINCT `cod_parada`) AS `cod_parada`,
         COUNT(DISTINCT `cod_controle_confirmacao`) AS `cod_controle_confirmacao`,
         COUNT(DISTINCT `dh_carga`) AS `dh_carga`
  FROM d GROUP BY `cod_requisicao_compra`, `cod_material`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(65,
    'cod_item_requisicao_compra', MAX(`cod_item_requisicao_compra`),
    'tp_documento_requisicao_compra', MAX(`tp_documento_requisicao_compra`),
    'st_processamento_requisicao_compra', MAX(`st_processamento_requisicao_compra`),
    'st_liberacao_requisicao_compra', MAX(`st_liberacao_requisicao_compra`),
    'ind_requisicao_compra_concluida', MAX(`ind_requisicao_compra_concluida`),
    'tp_ctg_item_documento_compra', MAX(`tp_ctg_item_documento_compra`),
    'num_pedido_compra', MAX(`num_pedido_compra`),
    'cod_centro', MAX(`cod_centro`),
    'num_item_pedido_compra', MAX(`num_item_pedido_compra`),
    'nm_centro', MAX(`nm_centro`),
    'cod_grupo_compradores', MAX(`cod_grupo_compradores`),
    'nm_grupo_compradores', MAX(`nm_grupo_compradores`),
    'cod_organizacao_compras', MAX(`cod_organizacao_compras`),
    'cod_texto_breve', MAX(`cod_texto_breve`),
    'desc_texto_longo', MAX(`desc_texto_longo`),
    'qt_solicitada', MAX(`qt_solicitada`),
    'sg_medida_pedido', MAX(`sg_medida_pedido`),
    'tp_grupo_mercadorias', MAX(`tp_grupo_mercadorias`),
    'vl_requisicao_compra', MAX(`vl_requisicao_compra`),
    'qt_disponibilidade_estoque', MAX(`qt_disponibilidade_estoque`),
    'tp_classificacao_contabil', MAX(`tp_classificacao_contabil`),
    'tp_material', MAX(`tp_material`),
    'vl_prazo_entrega_previsto_dias', MAX(`vl_prazo_entrega_previsto_dias`),
    'tp_mrp', MAX(`tp_mrp`),
    'cod_planejador_mrp', MAX(`cod_planejador_mrp`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'cod_elemento_pep', MAX(`cod_elemento_pep`),
    'cod_diagrama_rede', MAX(`cod_diagrama_rede`),
    'cod_conta_razao', MAX(`cod_conta_razao`),
    'num_reserva', MAX(`num_reserva`),
    'dt_solicitacao_rc', MAX(`dt_solicitacao_rc`),
    'dt_necessidade_rc', MAX(`dt_necessidade_rc`),
    'dt_liberacao_requisicao_compra', MAX(`dt_liberacao_requisicao_compra`),
    'qt_dias_rc', MAX(`qt_dias_rc`),
    'cod_usuario_criacao', MAX(`cod_usuario_criacao`),
    'nm_usuario_criacao', MAX(`nm_usuario_criacao`),
    'num_folha_servico', MAX(`num_folha_servico`),
    'dt_liberacao_folha', MAX(`dt_liberacao_folha`),
    'cod_servico', MAX(`cod_servico`),
    'desc_servico', MAX(`desc_servico`),
    'sg_medida_servico', MAX(`sg_medida_servico`),
    'dh_resposta', MAX(`dh_resposta`),
    'nm_motivo_rejeicao', MAX(`nm_motivo_rejeicao`),
    'desc_comentarios_fornecedor', MAX(`desc_comentarios_fornecedor`),
    'num_confirmacao_ordem', MAX(`num_confirmacao_ordem`),
    'qt_pedido', MAX(`qt_pedido`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'dt_pedido', MAX(`dt_pedido`),
    'cod_usuario_3', MAX(`cod_usuario_3`),
    'num_contrato', MAX(`num_contrato`),
    'ind_escopo_guiado', MAX(`ind_escopo_guiado`),
    'num_item_contrato', MAX(`num_item_contrato`),
    'dt_parada_inicio', MAX(`dt_parada_inicio`),
    'dt_parada_fim', MAX(`dt_parada_fim`),
    'num_referencia', MAX(`num_referencia`),
    'desc_anotacoes_comprador', MAX(`desc_anotacoes_comprador`),
    'ind_item_eliminado_rc', MAX(`ind_item_eliminado_rc`),
    'cod_eliminacao_documento_compras', MAX(`cod_eliminacao_documento_compras`),
    'ind_pedido_memorizado', MAX(`ind_pedido_memorizado`),
    'tp_urgencia', MAX(`tp_urgencia`),
    'ind_item_cotacao', MAX(`ind_item_cotacao`),
    'dt_envio_portal', MAX(`dt_envio_portal`),
    'cod_parada', MAX(`cod_parada`),
    'cod_controle_confirmacao', MAX(`cod_controle_confirmacao`),
    'dh_carga', MAX(`dh_carga`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Em validação anterior, esta análise revelou 7 colunas 100% nulas no Datalake —
uma delas preenchida em 97,9% dos registros do SAP. Este erro **não aparece** em amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

> **32 colunas não têm comentário** no `DESCRIBE` desta tabela. Coluna sem descrição costuma ser a assinatura de campo nunca carregado — confira o `veredito` delas com atenção.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, 'zeros_ou_false' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra),
perf AS (
  SELECT stack(67,
    'cod_requisicao_compra', 'string', COUNT_IF(`cod_requisicao_compra` IS NULL), COUNT_IF(`cod_requisicao_compra` IS NOT NULL AND lower(trim(`cod_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'cod_item_requisicao_compra', 'string', COUNT_IF(`cod_item_requisicao_compra` IS NULL), COUNT_IF(`cod_item_requisicao_compra` IS NOT NULL AND lower(trim(`cod_item_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_item_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'tp_documento_requisicao_compra', 'string', COUNT_IF(`tp_documento_requisicao_compra` IS NULL), COUNT_IF(`tp_documento_requisicao_compra` IS NOT NULL AND lower(trim(`tp_documento_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_documento_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'st_processamento_requisicao_compra', 'string', COUNT_IF(`st_processamento_requisicao_compra` IS NULL), COUNT_IF(`st_processamento_requisicao_compra` IS NOT NULL AND lower(trim(`st_processamento_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_processamento_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'st_liberacao_requisicao_compra', 'string', COUNT_IF(`st_liberacao_requisicao_compra` IS NULL), COUNT_IF(`st_liberacao_requisicao_compra` IS NOT NULL AND lower(trim(`st_liberacao_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_liberacao_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'ind_requisicao_compra_concluida', 'boolean', COUNT_IF(`ind_requisicao_compra_concluida` IS NULL), 0L, COUNT_IF(`ind_requisicao_compra_concluida` = false),
    'tp_ctg_item_documento_compra', 'string', COUNT_IF(`tp_ctg_item_documento_compra` IS NULL), COUNT_IF(`tp_ctg_item_documento_compra` IS NOT NULL AND lower(trim(`tp_ctg_item_documento_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_ctg_item_documento_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_pedido_compra', 'string', COUNT_IF(`num_pedido_compra` IS NULL), COUNT_IF(`num_pedido_compra` IS NOT NULL AND lower(trim(`num_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_compra', 'string', COUNT_IF(`num_item_pedido_compra` IS NULL), COUNT_IF(`num_item_pedido_compra` IS NOT NULL AND lower(trim(`num_item_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'nm_centro', 'string', COUNT_IF(`nm_centro` IS NULL), COUNT_IF(`nm_centro` IS NOT NULL AND lower(trim(`nm_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_compradores', 'string', COUNT_IF(`cod_grupo_compradores` IS NULL), COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND lower(trim(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'nm_grupo_compradores', 'string', COUNT_IF(`nm_grupo_compradores` IS NULL), COUNT_IF(`nm_grupo_compradores` IS NOT NULL AND lower(trim(`nm_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'cod_organizacao_compras', 'string', COUNT_IF(`cod_organizacao_compras` IS NULL), COUNT_IF(`cod_organizacao_compras` IS NOT NULL AND lower(trim(`cod_organizacao_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_organizacao_compras`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_texto_breve', 'string', COUNT_IF(`cod_texto_breve` IS NULL), COUNT_IF(`cod_texto_breve` IS NOT NULL AND lower(trim(`cod_texto_breve`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_texto_breve`) RLIKE '^0+([.,]0+)?$'),
    'desc_texto_longo', 'string', COUNT_IF(`desc_texto_longo` IS NULL), COUNT_IF(`desc_texto_longo` IS NOT NULL AND lower(trim(`desc_texto_longo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_texto_longo`) RLIKE '^0+([.,]0+)?$'),
    'qt_solicitada', 'decimal(13,3)', COUNT_IF(`qt_solicitada` IS NULL), 0L, COUNT_IF(`qt_solicitada` = 0),
    'sg_medida_pedido', 'string', COUNT_IF(`sg_medida_pedido` IS NULL), COUNT_IF(`sg_medida_pedido` IS NOT NULL AND lower(trim(`sg_medida_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_medida_pedido`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_mercadorias', 'string', COUNT_IF(`tp_grupo_mercadorias` IS NULL), COUNT_IF(`tp_grupo_mercadorias` IS NOT NULL AND lower(trim(`tp_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$'),
    'vl_requisicao_compra', 'double', COUNT_IF(`vl_requisicao_compra` IS NULL), 0L, COUNT_IF(`vl_requisicao_compra` = 0),
    'qt_disponibilidade_estoque', 'decimal(13,3)', COUNT_IF(`qt_disponibilidade_estoque` IS NULL), 0L, COUNT_IF(`qt_disponibilidade_estoque` = 0),
    'tp_classificacao_contabil', 'string', COUNT_IF(`tp_classificacao_contabil` IS NULL), COUNT_IF(`tp_classificacao_contabil` IS NOT NULL AND lower(trim(`tp_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_classificacao_contabil`) RLIKE '^0+([.,]0+)?$'),
    'tp_material', 'string', COUNT_IF(`tp_material` IS NULL), COUNT_IF(`tp_material` IS NOT NULL AND lower(trim(`tp_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_material`) RLIKE '^0+([.,]0+)?$'),
    'vl_prazo_entrega_previsto_dias', 'decimal(3,0)', COUNT_IF(`vl_prazo_entrega_previsto_dias` IS NULL), 0L, COUNT_IF(`vl_prazo_entrega_previsto_dias` = 0),
    'tp_mrp', 'string', COUNT_IF(`tp_mrp` IS NULL), COUNT_IF(`tp_mrp` IS NOT NULL AND lower(trim(`tp_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_planejador_mrp', 'string', COUNT_IF(`cod_planejador_mrp` IS NULL), COUNT_IF(`cod_planejador_mrp` IS NOT NULL AND lower(trim(`cod_planejador_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_planejador_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'cod_elemento_pep', 'string', COUNT_IF(`cod_elemento_pep` IS NULL), COUNT_IF(`cod_elemento_pep` IS NOT NULL AND lower(trim(`cod_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_elemento_pep`) RLIKE '^0+([.,]0+)?$'),
    'cod_diagrama_rede', 'string', COUNT_IF(`cod_diagrama_rede` IS NULL), COUNT_IF(`cod_diagrama_rede` IS NOT NULL AND lower(trim(`cod_diagrama_rede`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_diagrama_rede`) RLIKE '^0+([.,]0+)?$'),
    'cod_conta_razao', 'string', COUNT_IF(`cod_conta_razao` IS NULL), COUNT_IF(`cod_conta_razao` IS NOT NULL AND lower(trim(`cod_conta_razao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_conta_razao`) RLIKE '^0+([.,]0+)?$'),
    'num_reserva', 'string', COUNT_IF(`num_reserva` IS NULL), COUNT_IF(`num_reserva` IS NOT NULL AND lower(trim(`num_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_reserva`) RLIKE '^0+([.,]0+)?$'),
    'dt_solicitacao_rc', 'date', COUNT_IF(`dt_solicitacao_rc` IS NULL), 0L, 0L,
    'dt_necessidade_rc', 'date', COUNT_IF(`dt_necessidade_rc` IS NULL), 0L, 0L,
    'dt_liberacao_requisicao_compra', 'date', COUNT_IF(`dt_liberacao_requisicao_compra` IS NULL), 0L, 0L,
    'qt_dias_rc', 'int', COUNT_IF(`qt_dias_rc` IS NULL), 0L, COUNT_IF(`qt_dias_rc` = 0),
    'cod_usuario_criacao', 'string', COUNT_IF(`cod_usuario_criacao` IS NULL), COUNT_IF(`cod_usuario_criacao` IS NOT NULL AND lower(trim(`cod_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario_criacao', 'string', COUNT_IF(`nm_usuario_criacao` IS NULL), COUNT_IF(`nm_usuario_criacao` IS NOT NULL AND lower(trim(`nm_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'num_folha_servico', 'string', COUNT_IF(`num_folha_servico` IS NULL), COUNT_IF(`num_folha_servico` IS NOT NULL AND lower(trim(`num_folha_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_folha_servico`) RLIKE '^0+([.,]0+)?$'),
    'dt_liberacao_folha', 'date', COUNT_IF(`dt_liberacao_folha` IS NULL), 0L, 0L,
    'cod_servico', 'string', COUNT_IF(`cod_servico` IS NULL), COUNT_IF(`cod_servico` IS NOT NULL AND lower(trim(`cod_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_servico`) RLIKE '^0+([.,]0+)?$'),
    'desc_servico', 'string', COUNT_IF(`desc_servico` IS NULL), COUNT_IF(`desc_servico` IS NOT NULL AND lower(trim(`desc_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_servico`) RLIKE '^0+([.,]0+)?$'),
    'sg_medida_servico', 'string', COUNT_IF(`sg_medida_servico` IS NULL), COUNT_IF(`sg_medida_servico` IS NOT NULL AND lower(trim(`sg_medida_servico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_medida_servico`) RLIKE '^0+([.,]0+)?$'),
    'dh_resposta', 'timestamp', COUNT_IF(`dh_resposta` IS NULL), 0L, 0L,
    'nm_motivo_rejeicao', 'string', COUNT_IF(`nm_motivo_rejeicao` IS NULL), COUNT_IF(`nm_motivo_rejeicao` IS NOT NULL AND lower(trim(`nm_motivo_rejeicao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_motivo_rejeicao`) RLIKE '^0+([.,]0+)?$'),
    'desc_comentarios_fornecedor', 'string', COUNT_IF(`desc_comentarios_fornecedor` IS NULL), COUNT_IF(`desc_comentarios_fornecedor` IS NOT NULL AND lower(trim(`desc_comentarios_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_comentarios_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'num_confirmacao_ordem', 'string', COUNT_IF(`num_confirmacao_ordem` IS NULL), COUNT_IF(`num_confirmacao_ordem` IS NOT NULL AND lower(trim(`num_confirmacao_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_confirmacao_ordem`) RLIKE '^0+([.,]0+)?$'),
    'qt_pedido', 'decimal(13,3)', COUNT_IF(`qt_pedido` IS NULL), 0L, COUNT_IF(`qt_pedido` = 0),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'dt_pedido', 'date', COUNT_IF(`dt_pedido` IS NULL), 0L, 0L,
    'cod_usuario_3', 'string', COUNT_IF(`cod_usuario_3` IS NULL), COUNT_IF(`cod_usuario_3` IS NOT NULL AND lower(trim(`cod_usuario_3`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_3`) RLIKE '^0+([.,]0+)?$'),
    'num_contrato', 'string', COUNT_IF(`num_contrato` IS NULL), COUNT_IF(`num_contrato` IS NOT NULL AND lower(trim(`num_contrato`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_contrato`) RLIKE '^0+([.,]0+)?$'),
    'ind_escopo_guiado', 'boolean', COUNT_IF(`ind_escopo_guiado` IS NULL), 0L, COUNT_IF(`ind_escopo_guiado` = false),
    'num_item_contrato', 'string', COUNT_IF(`num_item_contrato` IS NULL), COUNT_IF(`num_item_contrato` IS NOT NULL AND lower(trim(`num_item_contrato`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_contrato`) RLIKE '^0+([.,]0+)?$'),
    'dt_parada_inicio', 'date', COUNT_IF(`dt_parada_inicio` IS NULL), 0L, 0L,
    'dt_parada_fim', 'date', COUNT_IF(`dt_parada_fim` IS NULL), 0L, 0L,
    'num_referencia', 'string', COUNT_IF(`num_referencia` IS NULL), COUNT_IF(`num_referencia` IS NOT NULL AND lower(trim(`num_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_referencia`) RLIKE '^0+([.,]0+)?$'),
    'desc_anotacoes_comprador', 'string', COUNT_IF(`desc_anotacoes_comprador` IS NULL), COUNT_IF(`desc_anotacoes_comprador` IS NOT NULL AND lower(trim(`desc_anotacoes_comprador`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_anotacoes_comprador`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_eliminado_rc', 'boolean', COUNT_IF(`ind_item_eliminado_rc` IS NULL), 0L, COUNT_IF(`ind_item_eliminado_rc` = false),
    'cod_eliminacao_documento_compras', 'string', COUNT_IF(`cod_eliminacao_documento_compras` IS NULL), COUNT_IF(`cod_eliminacao_documento_compras` IS NOT NULL AND lower(trim(`cod_eliminacao_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_eliminacao_documento_compras`) RLIKE '^0+([.,]0+)?$'),
    'ind_pedido_memorizado', 'boolean', COUNT_IF(`ind_pedido_memorizado` IS NULL), 0L, COUNT_IF(`ind_pedido_memorizado` = false),
    'tp_urgencia', 'string', COUNT_IF(`tp_urgencia` IS NULL), COUNT_IF(`tp_urgencia` IS NOT NULL AND lower(trim(`tp_urgencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_urgencia`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_cotacao', 'boolean', COUNT_IF(`ind_item_cotacao` IS NULL), 0L, COUNT_IF(`ind_item_cotacao` = false),
    'dt_envio_portal', 'date', COUNT_IF(`dt_envio_portal` IS NULL), 0L, 0L,
    'cod_parada', 'string', COUNT_IF(`cod_parada` IS NULL), COUNT_IF(`cod_parada` IS NOT NULL AND lower(trim(`cod_parada`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_parada`) RLIKE '^0+([.,]0+)?$'),
    'cod_controle_confirmacao', 'string', COUNT_IF(`cod_controle_confirmacao` IS NULL), COUNT_IF(`cod_controle_confirmacao` IS NOT NULL AND lower(trim(`cod_controle_confirmacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_controle_confirmacao`) RLIKE '^0+([.,]0+)?$'),
    'dh_carga', 'timestamp', COUNT_IF(`dh_carga` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra),
card AS (
  SELECT stack(67,
    'cod_requisicao_compra', 'string', approx_count_distinct(`cod_requisicao_compra`),
    'cod_item_requisicao_compra', 'string', approx_count_distinct(`cod_item_requisicao_compra`),
    'tp_documento_requisicao_compra', 'string', approx_count_distinct(`tp_documento_requisicao_compra`),
    'st_processamento_requisicao_compra', 'string', approx_count_distinct(`st_processamento_requisicao_compra`),
    'st_liberacao_requisicao_compra', 'string', approx_count_distinct(`st_liberacao_requisicao_compra`),
    'ind_requisicao_compra_concluida', 'boolean', approx_count_distinct(`ind_requisicao_compra_concluida`),
    'tp_ctg_item_documento_compra', 'string', approx_count_distinct(`tp_ctg_item_documento_compra`),
    'num_pedido_compra', 'string', approx_count_distinct(`num_pedido_compra`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'num_item_pedido_compra', 'string', approx_count_distinct(`num_item_pedido_compra`),
    'nm_centro', 'string', approx_count_distinct(`nm_centro`),
    'cod_grupo_compradores', 'string', approx_count_distinct(`cod_grupo_compradores`),
    'nm_grupo_compradores', 'string', approx_count_distinct(`nm_grupo_compradores`),
    'cod_organizacao_compras', 'string', approx_count_distinct(`cod_organizacao_compras`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_texto_breve', 'string', approx_count_distinct(`cod_texto_breve`),
    'desc_texto_longo', 'string', approx_count_distinct(`desc_texto_longo`),
    'qt_solicitada', 'decimal(13,3)', approx_count_distinct(`qt_solicitada`),
    'sg_medida_pedido', 'string', approx_count_distinct(`sg_medida_pedido`),
    'tp_grupo_mercadorias', 'string', approx_count_distinct(`tp_grupo_mercadorias`),
    'vl_requisicao_compra', 'double', approx_count_distinct(`vl_requisicao_compra`),
    'qt_disponibilidade_estoque', 'decimal(13,3)', approx_count_distinct(`qt_disponibilidade_estoque`),
    'tp_classificacao_contabil', 'string', approx_count_distinct(`tp_classificacao_contabil`),
    'tp_material', 'string', approx_count_distinct(`tp_material`),
    'vl_prazo_entrega_previsto_dias', 'decimal(3,0)', approx_count_distinct(`vl_prazo_entrega_previsto_dias`),
    'tp_mrp', 'string', approx_count_distinct(`tp_mrp`),
    'cod_planejador_mrp', 'string', approx_count_distinct(`cod_planejador_mrp`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'cod_elemento_pep', 'string', approx_count_distinct(`cod_elemento_pep`),
    'cod_diagrama_rede', 'string', approx_count_distinct(`cod_diagrama_rede`),
    'cod_conta_razao', 'string', approx_count_distinct(`cod_conta_razao`),
    'num_reserva', 'string', approx_count_distinct(`num_reserva`),
    'dt_solicitacao_rc', 'date', approx_count_distinct(`dt_solicitacao_rc`),
    'dt_necessidade_rc', 'date', approx_count_distinct(`dt_necessidade_rc`),
    'dt_liberacao_requisicao_compra', 'date', approx_count_distinct(`dt_liberacao_requisicao_compra`),
    'qt_dias_rc', 'int', approx_count_distinct(`qt_dias_rc`),
    'cod_usuario_criacao', 'string', approx_count_distinct(`cod_usuario_criacao`),
    'nm_usuario_criacao', 'string', approx_count_distinct(`nm_usuario_criacao`),
    'num_folha_servico', 'string', approx_count_distinct(`num_folha_servico`),
    'dt_liberacao_folha', 'date', approx_count_distinct(`dt_liberacao_folha`),
    'cod_servico', 'string', approx_count_distinct(`cod_servico`),
    'desc_servico', 'string', approx_count_distinct(`desc_servico`),
    'sg_medida_servico', 'string', approx_count_distinct(`sg_medida_servico`),
    'dh_resposta', 'timestamp', approx_count_distinct(`dh_resposta`),
    'nm_motivo_rejeicao', 'string', approx_count_distinct(`nm_motivo_rejeicao`),
    'desc_comentarios_fornecedor', 'string', approx_count_distinct(`desc_comentarios_fornecedor`),
    'num_confirmacao_ordem', 'string', approx_count_distinct(`num_confirmacao_ordem`),
    'qt_pedido', 'decimal(13,3)', approx_count_distinct(`qt_pedido`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'dt_pedido', 'date', approx_count_distinct(`dt_pedido`),
    'cod_usuario_3', 'string', approx_count_distinct(`cod_usuario_3`),
    'num_contrato', 'string', approx_count_distinct(`num_contrato`),
    'ind_escopo_guiado', 'boolean', approx_count_distinct(`ind_escopo_guiado`),
    'num_item_contrato', 'string', approx_count_distinct(`num_item_contrato`),
    'dt_parada_inicio', 'date', approx_count_distinct(`dt_parada_inicio`),
    'dt_parada_fim', 'date', approx_count_distinct(`dt_parada_fim`),
    'num_referencia', 'string', approx_count_distinct(`num_referencia`),
    'desc_anotacoes_comprador', 'string', approx_count_distinct(`desc_anotacoes_comprador`),
    'ind_item_eliminado_rc', 'boolean', approx_count_distinct(`ind_item_eliminado_rc`),
    'cod_eliminacao_documento_compras', 'string', approx_count_distinct(`cod_eliminacao_documento_compras`),
    'ind_pedido_memorizado', 'boolean', approx_count_distinct(`ind_pedido_memorizado`),
    'tp_urgencia', 'string', approx_count_distinct(`tp_urgencia`),
    'ind_item_cotacao', 'boolean', approx_count_distinct(`ind_item_cotacao`),
    'dt_envio_portal', 'date', approx_count_distinct(`dt_envio_portal`),
    'cod_parada', 'string', approx_count_distinct(`cod_parada`),
    'cod_controle_confirmacao', 'string', approx_count_distinct(`cod_controle_confirmacao`),
    'dh_carga', 'timestamp', approx_count_distinct(`dh_carga`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS (top 8 de cada)
(SELECT 'tp_documento_requisicao_compra' AS coluna, CAST(`tp_documento_requisicao_compra` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `tp_documento_requisicao_compra` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_processamento_requisicao_compra' AS coluna, CAST(`st_processamento_requisicao_compra` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `st_processamento_requisicao_compra` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_liberacao_requisicao_compra' AS coluna, CAST(`st_liberacao_requisicao_compra` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `st_liberacao_requisicao_compra` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_ctg_item_documento_compra' AS coluna, CAST(`tp_ctg_item_documento_compra` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `tp_ctg_item_documento_compra` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `cod_grupo_compradores` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_organizacao_compras' AS coluna, CAST(`cod_organizacao_compras` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `cod_organizacao_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_mercadorias' AS coluna, CAST(`tp_grupo_mercadorias` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `tp_grupo_mercadorias` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_classificacao_contabil' AS coluna, CAST(`tp_classificacao_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `tp_classificacao_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_material' AS coluna, CAST(`tp_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `tp_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_mrp' AS coluna, CAST(`tp_mrp` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `tp_mrp` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_urgencia' AS coluna, CAST(`tp_urgencia` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `tp_urgencia` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_eliminacao_documento_compras' AS coluna, CAST(`cod_eliminacao_documento_compras` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `cod_eliminacao_documento_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_medida_pedido' AS coluna, CAST(`sg_medida_pedido` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY `sg_medida_pedido` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Flags booleanas

A tabela tem 5 colunas `boolean`. Flag com valor único é candidata a
nunca ter sido carregada.

> **Atenção na comparação:** o SAP exporta indicadores como `X` / vazio;
> o Datalake grava `true` / `false`. É conversão de tipo, não divergência de valor.

In [0]:
-- FLAGS BOOLEANAS
SELECT 'ind_requisicao_compra_concluida' AS flag,
       COUNT_IF(`ind_requisicao_compra_concluida` = true)  AS verdadeiro,
       COUNT_IF(`ind_requisicao_compra_concluida` = false) AS falso,
       COUNT_IF(`ind_requisicao_compra_concluida` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_requisicao_compra_concluida` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'ind_escopo_guiado' AS flag,
       COUNT_IF(`ind_escopo_guiado` = true)  AS verdadeiro,
       COUNT_IF(`ind_escopo_guiado` = false) AS falso,
       COUNT_IF(`ind_escopo_guiado` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_escopo_guiado` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'ind_item_eliminado_rc' AS flag,
       COUNT_IF(`ind_item_eliminado_rc` = true)  AS verdadeiro,
       COUNT_IF(`ind_item_eliminado_rc` = false) AS falso,
       COUNT_IF(`ind_item_eliminado_rc` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_item_eliminado_rc` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'ind_pedido_memorizado' AS flag,
       COUNT_IF(`ind_pedido_memorizado` = true)  AS verdadeiro,
       COUNT_IF(`ind_pedido_memorizado` = false) AS falso,
       COUNT_IF(`ind_pedido_memorizado` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_pedido_memorizado` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'ind_item_cotacao' AS flag,
       COUNT_IF(`ind_item_cotacao` = true)  AS verdadeiro,
       COUNT_IF(`ind_item_cotacao` = false) AS falso,
       COUNT_IF(`ind_item_cotacao` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_item_cotacao` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
ORDER BY pct_verdadeiro DESC;

## 10. Perfil dos campos numéricos

Entre as colunas analisadas, **1 são `double`** — exigem tolerância de 0,005 na comparação com o SAP.

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(6,
    'qt_solicitada', 'decimal(13,3)', COUNT(`qt_solicitada`), CAST(MIN(`qt_solicitada`) AS DOUBLE), CAST(MAX(`qt_solicitada`) AS DOUBLE), CAST(AVG(`qt_solicitada`) AS DOUBLE), CAST(percentile_approx(`qt_solicitada`, 0.5) AS DOUBLE), COUNT_IF(`qt_solicitada` < 0), COUNT_IF(`qt_solicitada` = 0),
    'qt_disponibilidade_estoque', 'decimal(13,3)', COUNT(`qt_disponibilidade_estoque`), CAST(MIN(`qt_disponibilidade_estoque`) AS DOUBLE), CAST(MAX(`qt_disponibilidade_estoque`) AS DOUBLE), CAST(AVG(`qt_disponibilidade_estoque`) AS DOUBLE), CAST(percentile_approx(`qt_disponibilidade_estoque`, 0.5) AS DOUBLE), COUNT_IF(`qt_disponibilidade_estoque` < 0), COUNT_IF(`qt_disponibilidade_estoque` = 0),
    'vl_requisicao_compra', 'double', COUNT(`vl_requisicao_compra`), CAST(MIN(`vl_requisicao_compra`) AS DOUBLE), CAST(MAX(`vl_requisicao_compra`) AS DOUBLE), CAST(AVG(`vl_requisicao_compra`) AS DOUBLE), CAST(percentile_approx(`vl_requisicao_compra`, 0.5) AS DOUBLE), COUNT_IF(`vl_requisicao_compra` < 0), COUNT_IF(`vl_requisicao_compra` = 0),
    'vl_prazo_entrega_previsto_dias', 'decimal(3,0)', COUNT(`vl_prazo_entrega_previsto_dias`), CAST(MIN(`vl_prazo_entrega_previsto_dias`) AS DOUBLE), CAST(MAX(`vl_prazo_entrega_previsto_dias`) AS DOUBLE), CAST(AVG(`vl_prazo_entrega_previsto_dias`) AS DOUBLE), CAST(percentile_approx(`vl_prazo_entrega_previsto_dias`, 0.5) AS DOUBLE), COUNT_IF(`vl_prazo_entrega_previsto_dias` < 0), COUNT_IF(`vl_prazo_entrega_previsto_dias` = 0),
    'qt_dias_rc', 'int', COUNT(`qt_dias_rc`), CAST(MIN(`qt_dias_rc`) AS DOUBLE), CAST(MAX(`qt_dias_rc`) AS DOUBLE), CAST(AVG(`qt_dias_rc`) AS DOUBLE), CAST(percentile_approx(`qt_dias_rc`, 0.5) AS DOUBLE), COUNT_IF(`qt_dias_rc` < 0), COUNT_IF(`qt_dias_rc` = 0),
    'qt_pedido', 'decimal(13,3)', COUNT(`qt_pedido`), CAST(MIN(`qt_pedido`) AS DOUBLE), CAST(MAX(`qt_pedido`) AS DOUBLE), CAST(AVG(`qt_pedido`) AS DOUBLE), CAST(percentile_approx(`qt_pedido`, 0.5) AS DOUBLE), COUNT_IF(`qt_pedido` < 0), COUNT_IF(`qt_pedido` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(6,
    'qt_solicitada', CAST(SUM(`qt_solicitada`) AS DOUBLE), CAST(ROUND(SUM(`qt_solicitada`), 2) AS STRING), COUNT(`qt_solicitada`),
    'qt_disponibilidade_estoque', CAST(SUM(`qt_disponibilidade_estoque`) AS DOUBLE), CAST(ROUND(SUM(`qt_disponibilidade_estoque`), 2) AS STRING), COUNT(`qt_disponibilidade_estoque`),
    'vl_requisicao_compra', CAST(SUM(`vl_requisicao_compra`) AS DOUBLE), CAST(ROUND(SUM(`vl_requisicao_compra`), 2) AS STRING), COUNT(`vl_requisicao_compra`),
    'vl_prazo_entrega_previsto_dias', CAST(SUM(`vl_prazo_entrega_previsto_dias`) AS DOUBLE), CAST(ROUND(SUM(`vl_prazo_entrega_previsto_dias`), 2) AS STRING), COUNT(`vl_prazo_entrega_previsto_dias`),
    'qt_dias_rc', CAST(SUM(`qt_dias_rc`) AS DOUBLE), CAST(ROUND(SUM(`qt_dias_rc`), 2) AS STRING), COUNT(`qt_dias_rc`),
    'qt_pedido', CAST(SUM(`qt_pedido`) AS DOUBLE), CAST(ROUND(SUM(`qt_pedido`), 2) AS STRING), COUNT(`qt_pedido`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
)
ORDER BY coluna;

## 12. Datas

Todas as datas desta tabela estão em tipo nativo `date`. A coluna `sentinela_9999`
conta ocorrências de `9999-12-31`, usada pelo SAP para "sem prazo definido".

In [0]:
-- DATAS EM TIPO NATIVO
SELECT coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999
FROM (
  SELECT stack(8,
    'dt_solicitacao_rc', COUNT_IF(`dt_solicitacao_rc` IS NULL), CAST(MIN(`dt_solicitacao_rc`) AS STRING), CAST(MAX(`dt_solicitacao_rc`) AS STRING), COUNT(DISTINCT `dt_solicitacao_rc`), COUNT_IF(`dt_solicitacao_rc` > current_date()), COUNT_IF(`dt_solicitacao_rc` = DATE'9999-12-31'),
    'dt_necessidade_rc', COUNT_IF(`dt_necessidade_rc` IS NULL), CAST(MIN(`dt_necessidade_rc`) AS STRING), CAST(MAX(`dt_necessidade_rc`) AS STRING), COUNT(DISTINCT `dt_necessidade_rc`), COUNT_IF(`dt_necessidade_rc` > current_date()), COUNT_IF(`dt_necessidade_rc` = DATE'9999-12-31'),
    'dt_liberacao_requisicao_compra', COUNT_IF(`dt_liberacao_requisicao_compra` IS NULL), CAST(MIN(`dt_liberacao_requisicao_compra`) AS STRING), CAST(MAX(`dt_liberacao_requisicao_compra`) AS STRING), COUNT(DISTINCT `dt_liberacao_requisicao_compra`), COUNT_IF(`dt_liberacao_requisicao_compra` > current_date()), COUNT_IF(`dt_liberacao_requisicao_compra` = DATE'9999-12-31'),
    'dt_liberacao_folha', COUNT_IF(`dt_liberacao_folha` IS NULL), CAST(MIN(`dt_liberacao_folha`) AS STRING), CAST(MAX(`dt_liberacao_folha`) AS STRING), COUNT(DISTINCT `dt_liberacao_folha`), COUNT_IF(`dt_liberacao_folha` > current_date()), COUNT_IF(`dt_liberacao_folha` = DATE'9999-12-31'),
    'dt_pedido', COUNT_IF(`dt_pedido` IS NULL), CAST(MIN(`dt_pedido`) AS STRING), CAST(MAX(`dt_pedido`) AS STRING), COUNT(DISTINCT `dt_pedido`), COUNT_IF(`dt_pedido` > current_date()), COUNT_IF(`dt_pedido` = DATE'9999-12-31'),
    'dt_parada_inicio', COUNT_IF(`dt_parada_inicio` IS NULL), CAST(MIN(`dt_parada_inicio`) AS STRING), CAST(MAX(`dt_parada_inicio`) AS STRING), COUNT(DISTINCT `dt_parada_inicio`), COUNT_IF(`dt_parada_inicio` > current_date()), COUNT_IF(`dt_parada_inicio` = DATE'9999-12-31'),
    'dt_parada_fim', COUNT_IF(`dt_parada_fim` IS NULL), CAST(MIN(`dt_parada_fim`) AS STRING), CAST(MAX(`dt_parada_fim`) AS STRING), COUNT(DISTINCT `dt_parada_fim`), COUNT_IF(`dt_parada_fim` > current_date()), COUNT_IF(`dt_parada_fim` = DATE'9999-12-31'),
    'dt_envio_portal', COUNT_IF(`dt_envio_portal` IS NULL), CAST(MIN(`dt_envio_portal`) AS STRING), CAST(MAX(`dt_envio_portal`) AS STRING), COUNT(DISTINCT `dt_envio_portal`), COUNT_IF(`dt_envio_portal` > current_date()), COUNT_IF(`dt_envio_portal` = DATE'9999-12-31')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
)
ORDER BY coluna;

## 12.3 Timestamps

In [0]:
-- TIMESTAMPS
SELECT * FROM (
  SELECT stack(2,
    'dh_resposta', COUNT_IF(`dh_resposta` IS NULL), CAST(MIN(`dh_resposta`) AS STRING), CAST(MAX(`dh_resposta`) AS STRING), COUNT(DISTINCT `dh_resposta`),
    'dh_carga', COUNT_IF(`dh_carga` IS NULL), CAST(MIN(`dh_carga`) AS STRING), CAST(MAX(`dh_carga`) AS STRING), COUNT(DISTINCT `dh_carga`)
  ) AS (coluna, nulos, minimo, maximo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(11,
    'cod_requisicao_compra', 'string', COUNT_IF(CAST(`cod_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`cod_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`cod_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`cod_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`cod_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_requisicao_compra` AS STRING)), '^0+', '')),
    'cod_item_requisicao_compra', 'string', COUNT_IF(CAST(`cod_item_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`cod_item_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`cod_item_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`cod_item_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`cod_item_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_item_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_item_requisicao_compra` AS STRING)), '^0+', '')),
    'num_pedido_compra', 'string', COUNT_IF(CAST(`num_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '')),
    'num_item_pedido_compra', 'string', COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_item_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'num_contrato', 'string', COUNT_IF(CAST(`num_contrato` AS STRING) IS NULL OR trim(CAST(`num_contrato` AS STRING)) = ''), MIN(length(trim(CAST(`num_contrato` AS STRING)))), MAX(length(trim(CAST(`num_contrato` AS STRING)))), COUNT_IF(trim(CAST(`num_contrato` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_contrato` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_contrato` AS STRING)), '^0+', '')),
    'num_reserva', 'string', COUNT_IF(CAST(`num_reserva` AS STRING) IS NULL OR trim(CAST(`num_reserva` AS STRING)) = ''), MIN(length(trim(CAST(`num_reserva` AS STRING)))), MAX(length(trim(CAST(`num_reserva` AS STRING)))), COUNT_IF(trim(CAST(`num_reserva` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_reserva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_reserva` AS STRING)), '^0+', '')),
    'cod_conta_razao', 'string', COUNT_IF(CAST(`cod_conta_razao` AS STRING) IS NULL OR trim(CAST(`cod_conta_razao` AS STRING)) = ''), MIN(length(trim(CAST(`cod_conta_razao` AS STRING)))), MAX(length(trim(CAST(`cod_conta_razao` AS STRING)))), COUNT_IF(trim(CAST(`cod_conta_razao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_conta_razao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_conta_razao` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real,
não granularidade adicional.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_requisicao_compra` AS STRING), ''), COALESCE(CAST(`cod_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`tp_documento_requisicao_compra` AS STRING), ''), COALESCE(CAST(`st_processamento_requisicao_compra` AS STRING), ''), COALESCE(CAST(`st_liberacao_requisicao_compra` AS STRING), ''), COALESCE(CAST(`ind_requisicao_compra_concluida` AS STRING), ''), COALESCE(CAST(`tp_ctg_item_documento_compra` AS STRING), ''), COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_texto_breve` AS STRING), ''), COALESCE(CAST(`desc_texto_longo` AS STRING), ''), COALESCE(CAST(`qt_solicitada` AS STRING), ''), COALESCE(CAST(`sg_medida_pedido` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`vl_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_disponibilidade_estoque` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`vl_prazo_entrega_previsto_dias` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_planejador_mrp` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`dt_solicitacao_rc` AS STRING), ''), COALESCE(CAST(`dt_necessidade_rc` AS STRING), ''), COALESCE(CAST(`dt_liberacao_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_dias_rc` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`num_folha_servico` AS STRING), ''), COALESCE(CAST(`dt_liberacao_folha` AS STRING), ''), COALESCE(CAST(`cod_servico` AS STRING), ''), COALESCE(CAST(`desc_servico` AS STRING), ''), COALESCE(CAST(`sg_medida_servico` AS STRING), ''), COALESCE(CAST(`dh_resposta` AS STRING), ''), COALESCE(CAST(`nm_motivo_rejeicao` AS STRING), ''), COALESCE(CAST(`desc_comentarios_fornecedor` AS STRING), ''), COALESCE(CAST(`num_confirmacao_ordem` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`dt_pedido` AS STRING), ''), COALESCE(CAST(`cod_usuario_3` AS STRING), ''), COALESCE(CAST(`num_contrato` AS STRING), ''), COALESCE(CAST(`ind_escopo_guiado` AS STRING), ''), COALESCE(CAST(`num_item_contrato` AS STRING), ''), COALESCE(CAST(`dt_parada_inicio` AS STRING), ''), COALESCE(CAST(`dt_parada_fim` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`desc_anotacoes_comprador` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado_rc` AS STRING), ''), COALESCE(CAST(`cod_eliminacao_documento_compras` AS STRING), ''), COALESCE(CAST(`ind_pedido_memorizado` AS STRING), ''), COALESCE(CAST(`tp_urgencia` AS STRING), ''), COALESCE(CAST(`ind_item_cotacao` AS STRING), ''), COALESCE(CAST(`dt_envio_portal` AS STRING), ''), COALESCE(CAST(`cod_parada` AS STRING), ''), COALESCE(CAST(`cod_controle_confirmacao` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_requisicao_compra` AS STRING), ''), COALESCE(CAST(`cod_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`tp_documento_requisicao_compra` AS STRING), ''), COALESCE(CAST(`st_processamento_requisicao_compra` AS STRING), ''), COALESCE(CAST(`st_liberacao_requisicao_compra` AS STRING), ''), COALESCE(CAST(`ind_requisicao_compra_concluida` AS STRING), ''), COALESCE(CAST(`tp_ctg_item_documento_compra` AS STRING), ''), COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_texto_breve` AS STRING), ''), COALESCE(CAST(`desc_texto_longo` AS STRING), ''), COALESCE(CAST(`qt_solicitada` AS STRING), ''), COALESCE(CAST(`sg_medida_pedido` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`vl_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_disponibilidade_estoque` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`vl_prazo_entrega_previsto_dias` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_planejador_mrp` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`dt_solicitacao_rc` AS STRING), ''), COALESCE(CAST(`dt_necessidade_rc` AS STRING), ''), COALESCE(CAST(`dt_liberacao_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_dias_rc` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`num_folha_servico` AS STRING), ''), COALESCE(CAST(`dt_liberacao_folha` AS STRING), ''), COALESCE(CAST(`cod_servico` AS STRING), ''), COALESCE(CAST(`desc_servico` AS STRING), ''), COALESCE(CAST(`sg_medida_servico` AS STRING), ''), COALESCE(CAST(`dh_resposta` AS STRING), ''), COALESCE(CAST(`nm_motivo_rejeicao` AS STRING), ''), COALESCE(CAST(`desc_comentarios_fornecedor` AS STRING), ''), COALESCE(CAST(`num_confirmacao_ordem` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`dt_pedido` AS STRING), ''), COALESCE(CAST(`cod_usuario_3` AS STRING), ''), COALESCE(CAST(`num_contrato` AS STRING), ''), COALESCE(CAST(`ind_escopo_guiado` AS STRING), ''), COALESCE(CAST(`num_item_contrato` AS STRING), ''), COALESCE(CAST(`dt_parada_inicio` AS STRING), ''), COALESCE(CAST(`dt_parada_fim` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`desc_anotacoes_comprador` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado_rc` AS STRING), ''), COALESCE(CAST(`cod_eliminacao_documento_compras` AS STRING), ''), COALESCE(CAST(`ind_pedido_memorizado` AS STRING), ''), COALESCE(CAST(`tp_urgencia` AS STRING), ''), COALESCE(CAST(`ind_item_cotacao` AS STRING), ''), COALESCE(CAST(`dt_envio_portal` AS STRING), ''), COALESCE(CAST(`cod_parada` AS STRING), ''), COALESCE(CAST(`cod_controle_confirmacao` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_requisicao_compra` AS STRING), ''), COALESCE(CAST(`cod_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`tp_documento_requisicao_compra` AS STRING), ''), COALESCE(CAST(`st_processamento_requisicao_compra` AS STRING), ''), COALESCE(CAST(`st_liberacao_requisicao_compra` AS STRING), ''), COALESCE(CAST(`ind_requisicao_compra_concluida` AS STRING), ''), COALESCE(CAST(`tp_ctg_item_documento_compra` AS STRING), ''), COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_texto_breve` AS STRING), ''), COALESCE(CAST(`desc_texto_longo` AS STRING), ''), COALESCE(CAST(`qt_solicitada` AS STRING), ''), COALESCE(CAST(`sg_medida_pedido` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`vl_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_disponibilidade_estoque` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`vl_prazo_entrega_previsto_dias` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_planejador_mrp` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`dt_solicitacao_rc` AS STRING), ''), COALESCE(CAST(`dt_necessidade_rc` AS STRING), ''), COALESCE(CAST(`dt_liberacao_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_dias_rc` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`num_folha_servico` AS STRING), ''), COALESCE(CAST(`dt_liberacao_folha` AS STRING), ''), COALESCE(CAST(`cod_servico` AS STRING), ''), COALESCE(CAST(`desc_servico` AS STRING), ''), COALESCE(CAST(`sg_medida_servico` AS STRING), ''), COALESCE(CAST(`dh_resposta` AS STRING), ''), COALESCE(CAST(`nm_motivo_rejeicao` AS STRING), ''), COALESCE(CAST(`desc_comentarios_fornecedor` AS STRING), ''), COALESCE(CAST(`num_confirmacao_ordem` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`dt_pedido` AS STRING), ''), COALESCE(CAST(`cod_usuario_3` AS STRING), ''), COALESCE(CAST(`num_contrato` AS STRING), ''), COALESCE(CAST(`ind_escopo_guiado` AS STRING), ''), COALESCE(CAST(`num_item_contrato` AS STRING), ''), COALESCE(CAST(`dt_parada_inicio` AS STRING), ''), COALESCE(CAST(`dt_parada_fim` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`desc_anotacoes_comprador` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado_rc` AS STRING), ''), COALESCE(CAST(`cod_eliminacao_documento_compras` AS STRING), ''), COALESCE(CAST(`ind_pedido_memorizado` AS STRING), ''), COALESCE(CAST(`tp_urgencia` AS STRING), ''), COALESCE(CAST(`ind_item_cotacao` AS STRING), ''), COALESCE(CAST(`dt_envio_portal` AS STRING), ''), COALESCE(CAST(`cod_parada` AS STRING), ''), COALESCE(CAST(`cod_controle_confirmacao` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra;

## 15. Amostra de linhas completas

In [0]:
-- AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
ORDER BY `cod_requisicao_compra`, `cod_item_requisicao_compra`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_grupo_compradores
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)') AS `cod_grupo_compradores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_organizacao_compras
SELECT COALESCE(NULLIF(trim(CAST(`cod_organizacao_compras` AS STRING)), ''), '(vazio)') AS `cod_organizacao_compras`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_organizacao_compras` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR st_processamento_requisicao_compra
SELECT COALESCE(NULLIF(trim(CAST(`st_processamento_requisicao_compra` AS STRING)), ''), '(vazio)') AS `st_processamento_requisicao_compra`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY COALESCE(NULLIF(trim(CAST(`st_processamento_requisicao_compra` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_documento_requisicao_compra
SELECT COALESCE(NULLIF(trim(CAST(`tp_documento_requisicao_compra` AS STRING)), ''), '(vazio)') AS `tp_documento_requisicao_compra`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_documento_requisicao_compra` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

Esta tabela **não tem** coluna `dateingest`. Use os timestamps de carga.

In [0]:
-- 17. FRESHNESS PELOS TIMESTAMPS
SELECT MIN(dh_carga)                     AS carga_min,
       MAX(dh_carga)                     AS carga_max,
       COUNT(DISTINCT to_date(dh_carga)) AS dias_de_carga
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra;

## 18. Análises específicas — ZGLMM384

### 18.1 A chave inclui o item da requisição?

O clustering declara `cod_requisicao_compra + cod_material`, mas os comentários marcam
a PK como `cod_requisicao_compra + cod_item_requisicao_compra`.

Uma RC com vários itens quebra a unicidade do clustering.

In [0]:
-- 18.1 ITENS POR REQUISICAO
SELECT itens_na_requisicao, COUNT(*) AS requisicoes
FROM (SELECT cod_requisicao_compra,
             COUNT(DISTINCT cod_item_requisicao_compra) AS itens_na_requisicao
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra GROUP BY cod_requisicao_compra)
GROUP BY itens_na_requisicao
ORDER BY itens_na_requisicao
LIMIT 40;

In [0]:
-- 18.1b MATERIAL REPETIDO NA MESMA REQUISICAO
SELECT cod_requisicao_compra, cod_material,
       COUNT(DISTINCT cod_item_requisicao_compra) AS itens
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY cod_requisicao_compra, cod_material
HAVING COUNT(DISTINCT cod_item_requisicao_compra) > 1
ORDER BY itens DESC
LIMIT 20;

### 18.2 Ciclo RC → pedido

Nem toda requisição vira pedido. Esta célula mede a conversão e verifica se
quantidade pedida e solicitada são coerentes.

In [0]:
-- 18.2 CONVERSAO RC -> PEDIDO
SELECT COUNT(*)                                                       AS linhas,
       COUNT_IF(trim(COALESCE(num_pedido_compra,'')) <> '')           AS com_pedido,
       COUNT_IF(trim(COALESCE(num_pedido_compra,'')) = '')            AS sem_pedido,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(num_pedido_compra,'')) <> '') / COUNT(*), 2) AS pct_convertido,
       COUNT_IF(qt_pedido > qt_solicitada)                            AS pedido_maior_que_solicitado,
       COUNT_IF(qt_pedido < qt_solicitada)                            AS pedido_menor_que_solicitado,
       COUNT_IF(qt_pedido = qt_solicitada)                            AS quantidades_iguais,
       COUNT_IF(qt_pedido IS NULL)                                    AS sem_qt_pedido
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra;

### 18.3 Matriz de status e flags

Cruza as flags booleanas com o status de processamento para achar combinações
logicamente impossíveis.

In [0]:
-- 18.3 MATRIZ DE STATUS
SELECT COALESCE(NULLIF(trim(st_processamento_requisicao_compra), ''), '(vazio)') AS st_processamento,
       ind_requisicao_compra_concluida AS concluida,
       ind_item_eliminado_rc           AS eliminada,
       ind_item_cotacao                AS em_cotacao,
       COUNT(*)                        AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY COALESCE(NULLIF(trim(st_processamento_requisicao_compra), ''), '(vazio)'),
         ind_requisicao_compra_concluida, ind_item_eliminado_rc, ind_item_cotacao
ORDER BY linhas DESC
LIMIT 30;

In [0]:
-- 18.3b COMBINACOES SUSPEITAS
SELECT 'concluida MAS sem pedido de compra' AS situacao,
       COUNT_IF(ind_requisicao_compra_concluida = true
            AND trim(COALESCE(num_pedido_compra,'')) = '') AS linhas
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'eliminada MAS com pedido de compra',
       COUNT_IF(ind_item_eliminado_rc = true
            AND trim(COALESCE(num_pedido_compra,'')) <> '')
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'eliminada MAS sem codigo de eliminacao',
       COUNT_IF(ind_item_eliminado_rc = true
            AND trim(COALESCE(cod_eliminacao_documento_compras,'')) = '')
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'com folha de servico MAS sem codigo de servico',
       COUNT_IF(trim(COALESCE(num_folha_servico,'')) <> ''
            AND trim(COALESCE(cod_servico,'')) = '')
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
ORDER BY linhas DESC;

### 18.4 Coerência do fluxo de datas

Sequência esperada: solicitação → liberação → pedido → necessidade.

In [0]:
-- 18.4 COERENCIA DE DATAS
SELECT 'solicitacao <= liberacao' AS regra,
       COUNT_IF(dt_solicitacao_rc IS NOT NULL AND dt_liberacao_requisicao_compra IS NOT NULL) AS avaliadas,
       COUNT_IF(dt_solicitacao_rc > dt_liberacao_requisicao_compra) AS violacoes
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'liberacao <= pedido',
       COUNT_IF(dt_liberacao_requisicao_compra IS NOT NULL AND dt_pedido IS NOT NULL),
       COUNT_IF(dt_liberacao_requisicao_compra > dt_pedido)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'solicitacao <= necessidade',
       COUNT_IF(dt_solicitacao_rc IS NOT NULL AND dt_necessidade_rc IS NOT NULL),
       COUNT_IF(dt_solicitacao_rc > dt_necessidade_rc)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
UNION ALL
SELECT 'parada_inicio <= parada_fim',
       COUNT_IF(dt_parada_inicio IS NOT NULL AND dt_parada_fim IS NOT NULL),
       COUNT_IF(dt_parada_inicio > dt_parada_fim)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
ORDER BY violacoes DESC;

### 18.5 Classificação contábil

Cada RC aponta para centro de custo, elemento PEP, ordem ou rede.
Esta célula mostra a distribuição.

In [0]:
-- 18.5 CLASSIFICACAO CONTABIL
SELECT CASE WHEN trim(COALESCE(cod_centro_custo,'')) <> ''   THEN 'centro de custo'
            WHEN trim(COALESCE(cod_elemento_pep,'')) <> ''   THEN 'elemento PEP'
            WHEN trim(COALESCE(cod_diagrama_rede,'')) <> ''  THEN 'diagrama de rede'
            WHEN trim(COALESCE(num_reserva,'')) <> ''        THEN 'reserva'
            ELSE 'sem classificacao' END AS classificacao,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
GROUP BY 1
ORDER BY linhas DESC;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra
ORDER BY `cod_requisicao_compra`, `cod_item_requisicao_compra`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra),
g AS (
  SELECT 'cod_requisicao_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_requisicao_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
UNION ALL
  SELECT 'cod_requisicao_compra + cod_item_requisicao_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_requisicao_compra`, `cod_item_requisicao_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
UNION ALL
  SELECT 'cod_requisicao_compra + cod_item_requisicao_compra + num_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_requisicao_compra`, `cod_item_requisicao_compra`, `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
UNION ALL
  SELECT 'cod_requisicao_compra + cod_item_requisicao_compra + num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_requisicao_compra`, `cod_item_requisicao_compra`, `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm384_monitor_pedido_compra)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'ZGLMM384' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'tbl_ds_pro_zglmm384_monitor_pedido_compra'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '67'
UNION ALL SELECT 'IDENTIFICACAO', 'clustering declarado',
       'cod_requisicao_compra, cod_material'
UNION ALL SELECT 'IDENTIFICACAO', 'coluna de recorte', 'cod_centro'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Gerar o notebook de cenário com esse recorte.
3. Extrair a ZGLMM384 no SAP com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] Booleanas: `true`/`false` no Datalake vs `X`/vazio no SAP (seção 9)
- [ ] Alertas de partida verificados (seção 18)
